# Matrix Product States

In [1]:
using Tangles

In this example, we will be using the `MatrixProductState` or `MPS` type. In order to construct one, you can pass the arrays.

In [2]:
ψ = MPS([rand(ComplexF64, 2,2), rand(ComplexF64, 2, 2, 2), rand(ComplexF64, 2, 2)])

MPS (#tensors=3, #inds=5 #maxbondsize=2)

`CanonicalForm` or `form` returns the canonical form trait. Note that since we added the arrays directly, and there is no information about the orthogonality center, `MixedCanonicalMPS` interprets it as if the orthogonality center spans all the sites.

In [3]:
form(ψ)

NonCanonical()

In order to canonize it, you may call the `canonize!` function together with the `CanonicalForm` trait, which in this case should be `MixedCanonical(site_where_to_canonize)`.

In [4]:
canonize!(ψ, MixedCanonical(site"1"))

MPS (#tensors=3, #inds=5 #maxbondsize=2)

In [5]:
form(ψ)

MixedCanonical{CartesianSite{1}}(site<1>)

You can directly call `norm` and `normalize!` methods from `LinearAlgebra` too.

In [6]:
norm(ψ)

4.144782369132673

In [7]:
normalize!(ψ)

MPS (#tensors=3, #inds=5 #maxbondsize=2)

In [8]:
norm(ψ)

0.9999999999999999

Note that these methods may canonize or move the orthogonality center.

In [9]:
canonize!(ψ, MixedCanonical(all_sites(ψ)))
form(ψ)

MixedCanonical{Vector{Tangles.Site}}(Tangles.Site[site<1>, site<2>, site<3>])

In [10]:
norm(ψ)
form(ψ)

MixedCanonical{CartesianSite{1}}(site<1>)

## Time evolution

### Gate application using the "Simple Update" routine

In order to create an operator, you may use a regular `Tensor` but where the indices are `Index{Plug}`. This way, Tenet knows which `Tensor` indices connect with the `Tensor`s of the MPS.

In [11]:
op = NamedTensor(rand(ComplexF64, 2, 2, 2, 2), [Index(plug"2"), Index(plug"3"), Index(plug"2'"), Index(plug"3'")])

2×2×2×2 NamedTensor(::Tensor(::Array{ComplexF64, 4})) with signature ----:
[:, :, 1, 1] =
 0.206639+0.0901539im  0.909359+0.263955im
 0.371344+0.201802im    0.32468+0.205716im

[:, :, 2, 1] =
  0.24878+0.0967829im   0.741543+0.741623im
 0.375595+0.00278285im  0.386537+0.549797im

[:, :, 1, 2] =
 0.661693+0.961283im  0.990648+0.269372im
 0.372987+0.261197im  0.178761+0.383693im

[:, :, 2, 2] =
 0.189481+0.236966im  0.0251894+0.247693im
 0.748215+0.3479im     0.303716+0.961857im

In order to apply the operator, you can call the `simple_update!` function.

In [12]:
simple_update!(ψ, op)

MPS (#tensors=3, #inds=5 #maxbondsize=2)

As with the `norm` and `normalize!` functions, it canonizes to the acting sites of the gate for numerical precision.

In [13]:
form(ψ)

MixedCanonical{Vector{CartesianSite{1}}}(CartesianSite{1}[site<2>, site<3>])

### MPS-MPO contraction

In order to construct a `MatrixProductOperator` or `MPO`, you can directly pass the arrays or construct one random with `rand`.

In [14]:
mpo = rand(MPO; n=3, maxdim=4)

MatrixProductOperator (#tensors=3, #inds=8)

In order to contract the MPS and the MPO, use the general purpose `evolve!` function.

In [15]:
evolve!(ψ, mpo)

MPS (#tensors=3, #inds=5 #maxbondsize=8)

Note that, currently, the bond dimension of the MPS increases with the evolution.

In [16]:
for bond in all_bonds(ψ)
    println("bond = $bond --> size = $(size(ψ, ind_at(ψ, bond)))")
end

bond = bond<site<1> ⟷ site<2>> --> size = 8
bond = bond<site<2> ⟷ site<3>> --> size = 8


## Compression

In order to reduce the size of the virtual bonds, you may use the `compress!` function. It accepts `maxdim` and `threshold` (which truncates based on `abs`olute value of the singular values) kwargs.

In [17]:
compress!(ψ; maxdim=4)

MPS (#tensors=4, #inds=5 #maxbondsize=2)

In [18]:
for bond in all_bonds(ψ)
    println("bond = $bond --> size = $(size(ψ, ind_at(ψ, bond)))")
end

bond = bond<site<1> ⟷ site<2>> --> size = 2
bond = bond<site<2> ⟷ site<3>> --> size = 2
